In [ ]:
# 如果已经安装过，可以跳过这个 cell
!pip install sentence-transformers numpy tqdm pandas

In [20]:
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


# ===== 路径设置 =====
PROJECT_ROOT = Path(r"E:\CodexWorkspace\RAG_reading")
DATA_DIR = PROJECT_ROOT / "donnee" / "Download 2026-04-30T08-00-29-368Z"

EMBEDDINGS_PATH = DATA_DIR / "chunk_embeddings(multi).npy"
METADATA_PATH = DATA_DIR / "chunk_metadata.jsonl"

JSON_OUTPUT_PATH = PROJECT_ROOT / "rag_test_answers(official).json"
MARKDOWN_OUTPUT_PATH = PROJECT_ROOT / "rag_test_answers(official).md"


# ===== 模型与检索参数 =====
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

TOP_K = 5
HYBRID_CANDIDATES_K = 50

HYBRID_DENSE_WEIGHT = 0.5
HYBRID_BM25_WEIGHT = 0.5

BM25_K1 = 1.5
BM25_B = 0.75


# ===== 测试问题 =====
TEST_QUERIES = [
    {
        "qid": "q1",
        "question": "Comment l’intégration du drone MQ-9 Reaper dans l’Armée de l’air et de l’espace française se compare-t-elle à l’adoption potentielle du TEKEVER AR-3 pour des missions ISR (renseignement, surveillance et reconnaissance), en termes d’impact opérationnel et humain, notamment sur les besoins en équipages, la formation et la capacité d’évolution future des effectifs ?",
        "theme": "Drones / lutte anti-drones",
    },
    {
        "qid": "q2",
        "question": "Quel est l'objectif du projet \"Beehive\" de la Royal Navy, et quelles sont les caractéristiques clés des USV (véhicules de surface sans équipage) qu'elle souhaite acquérir dans ce cadre ?",
        "theme": "Protection du secret de la defense nationale",
    },
    {
        "qid": "q3",
        "question": "Comment les mesures anti-drones proposées en France, les capacités des munitions rôdeuses Anduril Altius 600M-V et AeroVironment Switchblade 300 vendues à Taïwan, et l’évolution de la stratégie navale chinoise face aux attaques d’USV ukrainiens illustrent-ils ensemble une tendance mondiale vers l’intégration de la guerre électronique, de la neutralisation des drones et des systèmes de défense multi-domaines ?",
        "theme": "Cybersecurite / securite numerique",
    },
    {
        "qid": "q4",
        "question": "Comment les méthodes de renseignement telles que l’OSINT et l’HUMINT pourraient-elles être intégrées aux missions de Tracfin ?",
        "theme": "Renseignement / surveillance / reconnaissance",
    },
    {
        "qid": "q5",
        "question": "Quels sont les drones capables de transporter et de larguer du matériel ?",
        "theme": "Organisation des forces armees / doctrine militaire",
    },
]

Cell 2：导入库与参数设置

In [21]:
def load_metadata(metadata_path):
    metadata = []

    with metadata_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                metadata.append(json.loads(line))

    return metadata


def get_text(item):
    """
    兼容不同字段名：
    - text
    - texte
    """
    return item.get("text") or item.get("texte") or ""


def get_doc_name(item):
    """
    兼容不同字段名：
    - doc_name
    - source
    """
    return item.get("doc_name") or item.get("source") or ""


def get_page(item):
    return item.get("page", None)


def get_chunk_id(item, idx=None):
    if "chunk_id" in item:
        return item["chunk_id"]

    doc_name = get_doc_name(item)
    page = get_page(item)
    return f"{doc_name}_page_{page}_idx_{idx}"


def tokenize_for_bm25(text):
    text = text.lower()
    return re.findall(r"(?u)\b\w+(?:[-']\w+)*\b", text)


def build_bm25_index(metadata):
    doc_term_freqs = []
    doc_lengths = []
    doc_freqs = defaultdict(int)

    for item in metadata:
        tokens = tokenize_for_bm25(get_text(item))
        term_freqs = Counter(tokens)

        doc_term_freqs.append(term_freqs)
        doc_lengths.append(len(tokens))

        for term in term_freqs:
            doc_freqs[term] += 1

    doc_count = len(metadata)
    avg_doc_length = sum(doc_lengths) / doc_count if doc_count else 0.0

    idf = {
        term: math.log(1 + (doc_count - freq + 0.5) / (freq + 0.5))
        for term, freq in doc_freqs.items()
    }

    inverted_index = defaultdict(list)

    for doc_idx, term_freqs in enumerate(doc_term_freqs):
        for term, freq in term_freqs.items():
            inverted_index[term].append((doc_idx, freq))

    return {
        "doc_count": doc_count,
        "avg_doc_length": avg_doc_length,
        "doc_lengths": doc_lengths,
        "idf": idf,
        "inverted_index": inverted_index,
        "k1": BM25_K1,
        "b": BM25_B,
    }


def load_existing_index():
    if not EMBEDDINGS_PATH.exists():
        raise FileNotFoundError(
            f"Embedding file not found:\n{EMBEDDINGS_PATH}\n\n"
            "请先运行生成 embedding index 的脚本。"
        )

    if not METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Metadata file not found:\n{METADATA_PATH}\n\n"
            "请先运行生成 chunk_metadata.jsonl 的脚本。"
        )

    print(f"Loading embeddings:\n{EMBEDDINGS_PATH}")
    embeddings = np.load(EMBEDDINGS_PATH)

    print(f"Loading metadata:\n{METADATA_PATH}")
    metadata = load_metadata(METADATA_PATH)

    if len(metadata) != len(embeddings):
        raise ValueError(
            f"Metadata count ({len(metadata)}) does not match embedding count "
            f"({len(embeddings)}). 请重新生成 embedding index。"
        )

    print(f"Loading embedding model:\n{MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)

    print("Building BM25 index...")
    bm25_index = build_bm25_index(metadata)

    print("Index ready.")
    print(f"Metadata records: {len(metadata)}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"BM25 documents: {bm25_index['doc_count']}")

    return model, embeddings, metadata, bm25_index

Cell 3：加载数据与构建 BM25

In [22]:
def load_metadata(metadata_path):
    metadata = []

    with metadata_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                metadata.append(json.loads(line))

    return metadata


def get_text(item):
    """
    兼容不同字段名：
    - text
    - texte
    """
    return item.get("text") or item.get("texte") or ""


def get_doc_name(item):
    """
    兼容不同字段名：
    - doc_name
    - source
    """
    return item.get("doc_name") or item.get("source") or ""


def get_page(item):
    return item.get("page", None)


def get_chunk_id(item, idx=None):
    if "chunk_id" in item:
        return item["chunk_id"]

    doc_name = get_doc_name(item)
    page = get_page(item)
    return f"{doc_name}_page_{page}_idx_{idx}"


def tokenize_for_bm25(text):
    text = text.lower()
    return re.findall(r"(?u)\b\w+(?:[-']\w+)*\b", text)


def build_bm25_index(metadata):
    doc_term_freqs = []
    doc_lengths = []
    doc_freqs = defaultdict(int)

    for item in metadata:
        tokens = tokenize_for_bm25(get_text(item))
        term_freqs = Counter(tokens)

        doc_term_freqs.append(term_freqs)
        doc_lengths.append(len(tokens))

        for term in term_freqs:
            doc_freqs[term] += 1

    doc_count = len(metadata)
    avg_doc_length = sum(doc_lengths) / doc_count if doc_count else 0.0

    idf = {
        term: math.log(1 + (doc_count - freq + 0.5) / (freq + 0.5))
        for term, freq in doc_freqs.items()
    }

    inverted_index = defaultdict(list)

    for doc_idx, term_freqs in enumerate(doc_term_freqs):
        for term, freq in term_freqs.items():
            inverted_index[term].append((doc_idx, freq))

    return {
        "doc_count": doc_count,
        "avg_doc_length": avg_doc_length,
        "doc_lengths": doc_lengths,
        "idf": idf,
        "inverted_index": inverted_index,
        "k1": BM25_K1,
        "b": BM25_B,
    }


def load_existing_index():
    if not EMBEDDINGS_PATH.exists():
        raise FileNotFoundError(
            f"Embedding file not found:\n{EMBEDDINGS_PATH}\n\n"
            "请先运行生成 embedding index 的脚本。"
        )

    if not METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Metadata file not found:\n{METADATA_PATH}\n\n"
            "请先运行生成 chunk_metadata.jsonl 的脚本。"
        )

    print(f"Loading embeddings:\n{EMBEDDINGS_PATH}")
    embeddings = np.load(EMBEDDINGS_PATH)

    print(f"Loading metadata:\n{METADATA_PATH}")
    metadata = load_metadata(METADATA_PATH)

    if len(metadata) != len(embeddings):
        raise ValueError(
            f"Metadata count ({len(metadata)}) does not match embedding count "
            f"({len(embeddings)}). 请重新生成 embedding index。"
        )

    print(f"Loading embedding model:\n{MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)

    print("Building BM25 index...")
    bm25_index = build_bm25_index(metadata)

    print("Index ready.")
    print(f"Metadata records: {len(metadata)}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"BM25 documents: {bm25_index['doc_count']}")

    return model, embeddings, metadata, bm25_index

Cell 4：定义 dense / BM25 / hybrid 检索函数

In [23]:
def dense_retrieve(query, model, embeddings, metadata, top_k=TOP_K):
    query_embedding = model.encode(query, normalize_embeddings=True)
    query_embedding = np.asarray(query_embedding, dtype="float32")

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, 1):
        idx = int(idx)
        item = metadata[idx]

        results.append(
            {
                "rank": rank,
                "score": float(scores[idx]),
                "chunk_id": get_chunk_id(item, idx),
                "doc_name": get_doc_name(item),
                "page": get_page(item),
                "text": get_text(item),
            }
        )

    return results


def bm25_retrieve(query, metadata, bm25_index, top_k=TOP_K):
    query_terms = tokenize_for_bm25(query)
    scores = defaultdict(float)
    avg_doc_length = bm25_index["avg_doc_length"]

    if not query_terms or not avg_doc_length:
        return []

    for term in query_terms:
        term_idf = bm25_index["idf"].get(term)

        if term_idf is None:
            continue

        for doc_idx, term_freq in bm25_index["inverted_index"].get(term, []):
            doc_length = bm25_index["doc_lengths"][doc_idx]

            denominator = (
                term_freq
                + bm25_index["k1"]
                * (
                    1
                    - bm25_index["b"]
                    + bm25_index["b"] * doc_length / avg_doc_length
                )
            )

            scores[doc_idx] += term_idf * (
                term_freq * (bm25_index["k1"] + 1) / denominator
            )

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

    results = []

    for rank, (idx, score) in enumerate(ranked, 1):
        idx = int(idx)
        item = metadata[idx]

        results.append(
            {
                "rank": rank,
                "score": float(score),
                "chunk_id": get_chunk_id(item, idx),
                "doc_name": get_doc_name(item),
                "page": get_page(item),
                "text": get_text(item),
            }
        )

    return results


def normalize_score_map(results):
    if not results:
        return {}

    scores = [item["score"] for item in results]

    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return {item["chunk_id"]: 1.0 for item in results}

    return {
        item["chunk_id"]: (item["score"] - min_score) / (max_score - min_score)
        for item in results
    }


def hybrid_retrieve(
    query,
    model,
    embeddings,
    metadata,
    bm25_index,
    top_k=TOP_K,
    candidate_k=HYBRID_CANDIDATES_K,
    dense_weight=HYBRID_DENSE_WEIGHT,
    bm25_weight=HYBRID_BM25_WEIGHT,
):
    dense_results = dense_retrieve(
        query,
        model=model,
        embeddings=embeddings,
        metadata=metadata,
        top_k=candidate_k,
    )

    bm25_results = bm25_retrieve(
        query,
        metadata=metadata,
        bm25_index=bm25_index,
        top_k=candidate_k,
    )

    dense_scores = normalize_score_map(dense_results)
    bm25_scores = normalize_score_map(bm25_results)

    merged = {}

    for source_name, source_results in [
        ("dense", dense_results),
        ("bm25", bm25_results),
    ]:
        for item in source_results:
            chunk_id = item["chunk_id"]

            if chunk_id not in merged:
                merged[chunk_id] = {
                    "chunk_id": chunk_id,
                    "doc_name": item["doc_name"],
                    "page": item["page"],
                    "text": item["text"],
                    "sources": [],
                }

            merged[chunk_id]["sources"].append(source_name)

    ranked = []

    for chunk_id, item in merged.items():
        dense_score = dense_scores.get(chunk_id, 0.0)
        bm25_score = bm25_scores.get(chunk_id, 0.0)

        final_score = dense_weight * dense_score + bm25_weight * bm25_score

        ranked.append(
            {
                "score": float(final_score),
                "dense_score": float(dense_score),
                "bm25_score": float(bm25_score),
                **item,
            }
        )

    ranked.sort(key=lambda item: item["score"], reverse=True)

    for rank, item in enumerate(ranked[:top_k], 1):
        item["rank"] = rank

    return ranked[:top_k]

Cell 5：运行测试查询并导出结果

In [24]:
def make_preview(text, max_chars=700):
    text = " ".join(text.split())

    if len(text) <= max_chars:
        return text

    return text[: max_chars - 3].rstrip() + "..."


def save_markdown(output, output_path):
    lines = [
        "# RAG Test Answers",
        "",
        f"Generated at: {output['generated_at']}",
        "",
        f"Retriever: {output['retriever']['type']}",
        f"Top K: {output['retriever']['top_k']}",
        f"Candidate K each: {output['retriever']['candidate_k_each']}",
        "",
    ]

    for query_result in output["results"]:
        lines.extend(
            [
                f"## {query_result['qid']} - {query_result['theme']}",
                "",
                f"Question: {query_result['question']}",
                "",
            ]
        )

        for item in query_result["answer"]:
            lines.extend(
                [
                    f"### Rank {item['rank']}",
                    "",
                    f"- Score: {item['score']:.4f}",
                    f"- Dense score: {item['dense_score']:.4f}",
                    f"- BM25 score: {item['bm25_score']:.4f}",
                    f"- Sources: {', '.join(item['sources'])}",
                    f"- Document: {item['doc_name']}",
                    f"- Page: {item['page']}",
                    f"- Chunk ID: {item['chunk_id']}",
                    "",
                    make_preview(item["text"]),
                    "",
                ]
            )

    output_path.write_text("\n".join(lines), encoding="utf-8")


def flatten_results_to_dataframe(output):
    rows = []

    for query_result in output["results"]:
        for item in query_result["answer"]:
            rows.append(
                {
                    "qid": query_result["qid"],
                    "theme": query_result["theme"],
                    "question": query_result["question"],
                    "rank": item["rank"],
                    "score": item["score"],
                    "dense_score": item["dense_score"],
                    "bm25_score": item["bm25_score"],
                    "sources": ", ".join(item["sources"]),
                    "doc_name": item["doc_name"],
                    "page": item["page"],
                    "chunk_id": item["chunk_id"],
                    "preview": make_preview(item["text"], max_chars=500),
                }
            )

    return pd.DataFrame(rows)


def run_rag_test_queries(
    top_k=TOP_K,
    candidate_k=HYBRID_CANDIDATES_K,
    json_output_path=JSON_OUTPUT_PATH,
    markdown_output_path=MARKDOWN_OUTPUT_PATH,
):
    model, embeddings, metadata, bm25_index = load_existing_index()

    output = {
        "run_id": "rag_test_queries_hybrid_v1",
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "retriever": {
            "type": "hybrid_bm25_dense",
            "dense_model_name": MODEL_NAME,
            "top_k": top_k,
            "candidate_k_each": candidate_k,
            "dense_weight": HYBRID_DENSE_WEIGHT,
            "bm25_weight": HYBRID_BM25_WEIGHT,
            "bm25": {
                "k1": BM25_K1,
                "b": BM25_B,
                "tokenizer": "unicode_words_keep_hyphenated_identifiers",
            },
            "embeddings_path": str(EMBEDDINGS_PATH),
            "metadata_path": str(METADATA_PATH),
        },
        "results": [],
    }

    for query in tqdm(TEST_QUERIES, desc="Running RAG test queries"):
        answer = hybrid_retrieve(
            query["question"],
            model=model,
            embeddings=embeddings,
            metadata=metadata,
            bm25_index=bm25_index,
            top_k=top_k,
            candidate_k=candidate_k,
        )

        output["results"].append(
            {
                "qid": query["qid"],
                "theme": query["theme"],
                "question": query["question"],
                "answer": answer,
            }
        )

    with json_output_path.open("w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    save_markdown(output, markdown_output_path)

    print(f"Saved JSON answers: {json_output_path}")
    print(f"Saved Markdown answers: {markdown_output_path}")

    df = flatten_results_to_dataframe(output)

    return output, df

Cell 6：实际运行

In [25]:
rag_test_output, df = run_rag_test_queries(
    top_k=5,
    candidate_k=50,
)

df

Loading embeddings:
E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_embeddings(multi).npy
Loading metadata:
E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_metadata.jsonl
Loading embedding model:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Building BM25 index...
Index ready.
Metadata records: 29202
Embedding shape: (29202, 384)
BM25 documents: 29202


Running RAG test queries:   0%|          | 0/5 [00:00<?, ?it/s]

Saved JSON answers: E:\CodexWorkspace\RAG_reading\rag_test_answers(official).json
Saved Markdown answers: E:\CodexWorkspace\RAG_reading\rag_test_answers(official).md


,qid,theme,question,rank,score,dense_score,bm25_score,sources,doc_name,page,chunk_id,preview
0,q1,Drones / lutte anti-drones,Comment l’intégration du drone MQ-9 Reaper dan...,1,0.866396,1.000000,0.732792,"dense, bm25",MQ-9 Reaper _ Ministère des Armées et des Anci...,1,MQ-9_Reaper_Minist_re_des_Arm_es_et_des_Ancien...,MQ-9 Reaper Équipements Aéronefs Drones Le MQ-...
1,q1,Drones / lutte anti-drones,Comment l’intégration du drone MQ-9 Reaper dan...,2,0.500000,0.000000,1.000000,bm25,"Le nouveau Reaper block 5 ER _ plus loin, plus...",4,Le_nouveau_Reaper_block_5_ER_plus_loin_plus_lo...,"Le nouveau Reaper block 5 ER : plus loin, plus..."
2,q1,Drones / lutte anti-drones,Comment l’intégration du drone MQ-9 Reaper dan...,3,0.491934,0.000000,0.983868,bm25,Salon du Bourget _ Interception coordonnée d’u...,3,Salon_du_Bourget_Interception_coordonn_e_d_un_...,analyse fine de la situation. Interception coo...
3,q1,Drones / lutte anti-drones,Comment l’intégration du drone MQ-9 Reaper dan...,4,0.476059,0.000000,0.952119,bm25,LA 33E ESRA ET LE REAPER EN MISSION DE SURVEIL...,2,LA_33E_ESRA_ET_LE_REAPER_EN_MISSION_DE_SURVEIL...,LA 33E ESRA ET LE REAPER EN MISSION DE SURVEIL...
4,q1,Drones / lutte anti-drones,Comment l’intégration du drone MQ-9 Reaper dan...,5,0.399257,0.216504,0.582010,"dense, bm25",Devenir pilote de drone militaire.pdf,2,Devenir_pilote_de_drone_militaire_p2_c000,Le poste Devenir pilote de drone pourra vous a...
5,q2,Protection du secret de la defense nationale,"Quel est l'objectif du projet ""Beehive"" de la ...",1,0.500000,1.000000,0.000000,dense,ObsDrones - bulletin de veille n°10 - juillet-...,14,ObsDrones_-_bulletin_de_veille_n_10_-_juillet-...,"rapide, dans un cadre programmatique en trois ..."
6,q2,Protection du secret de la defense nationale,"Quel est l'objectif du projet ""Beehive"" de la ...",2,0.500000,0.000000,1.000000,bm25,20260108_NP_Obsdrones_Bulletin-de-veille-n12_0...,17,20260108_NP_Obsdrones_Bulletin-de-veille-n12_0...,Le Corsair ASV a une CU de 453 kg sur 1 000 nq...
7,q2,Protection du secret de la defense nationale,"Quel est l'objectif du projet ""Beehive"" de la ...",3,0.492658,0.765087,0.220230,"dense, bm25",ObsDrones - Bulletin de veille n1 - Janvier Fé...,11,ObsDrones_-_Bulletin_de_veille_n1_-_Janvier_F_...,NAVAL L’US Navy vise le déploiement d’essaim d...
8,q2,Protection du secret de la defense nationale,"Quel est l'objectif du projet ""Beehive"" de la ...",4,0.297222,0.594444,0.000000,dense,ObsDrones - Bulletin de veille n1 - Janvier Fé...,11,ObsDrones_-_Bulletin_de_veille_n1_-_Janvier_F_...,de long). La marine portugaise a des projets a...
9,q2,Protection du secret de la defense nationale,"Quel est l'objectif du projet ""Beehive"" de la ...",5,0.273348,0.546696,0.000000,dense,Bulletin de veille d’actualité - Septembre 202...,14,Bulletin_de_veille_d_actualit_-_Septembre_2025...,des gouvernails en forme de X à l’arrière. L’u...


In [11]:
import numpy as np
import json

embeddings = np.load(r"E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_embeddings.npy")

print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(29202, 384)


In [12]:
metadata_path = r"E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_metadata.jsonl"

metadata = []

with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        metadata.append(json.loads(line))

print(len(metadata))
print(embeddings.shape[0])

29202
29202
